In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DA BASE TELESAP
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
df = pd.read_csv(file_path, low_memory=False)

# ==============================================================================
# 2. RASTREAMENTO DE TUBERCULOSE E HIV (TEXT MINING + CHECKBOXES)
# ==============================================================================
# Filtra colunas de texto para buscar anotações médicas
text_cols = [c for c in df.columns if df[c].dtype == object]
df_text = df[text_cols].fillna('')

# Busca nos textos livres
cond_tb_text = df_text.apply(lambda x: x.astype(str).str.contains(r'\b(tuberculose|tb)\b', case=False, regex=True)).any(axis=1)
cond_hiv_text = df_text.apply(lambda x: x.astype(str).str.contains(r'\b(hiv|aids|sida|vírus da imunodeficiência)\b', case=False, regex=True)).any(axis=1)

# Busca nas colunas estruturadas (Doenças Infecciosas ou CIAP)
cols_infecciosas = [c for c in df.columns if ('infeccios' in c.lower() or 'ciap' in c.lower() or 'choice=' in c.lower())]

cond_tb_str = pd.Series(False, index=df.index)
cond_hiv_str = pd.Series(False, index=df.index)

for c in cols_infecciosas:
    is_checked = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
    
    if 'tuberculose' in c.lower() or 'a70' in c.lower(): # A70 é o CIAP de Tuberculose
        cond_tb_str = cond_tb_str | is_checked
    if 'hiv' in c.lower() or 'aids' in c.lower() or 'b90' in c.lower(): # B90 é o CIAP de HIV
        cond_hiv_str = cond_hiv_str | is_checked

# Unifica as descobertas: 1 se tem a doença, 0 se não tem
df['Tem_TB'] = (cond_tb_text | cond_tb_str).astype(int)
df['Tem_HIV'] = (cond_hiv_text | cond_hiv_str).astype(int)

# ==============================================================================
# 3. CRIAÇÃO DA TABELA DESAGREGADA (TODOS OS DADOS POR PACIENTE)
# ==============================================================================
# Agrupa pelo ID do paciente. Se ele teve registro de TB/HIV em qualquer consulta, conta como 1
df_pacientes = df.groupby('Record ID')[['Tem_TB', 'Tem_HIV']].max().reset_index()

# Mapeia para os nomes que aparecerão no gráfico
df_pacientes['Grupo'] = df_pacientes['Tem_TB'].map({0: 'Sem Tuberculose', 1: 'Com Tuberculose'})
df_pacientes['Status_HIV'] = df_pacientes['Tem_HIV'].map({0: 'Negativo', 1: 'Positivo'})

# ---> EXPORTAÇÃO DA TABELA BRUTA <---
# Aqui está a tabela com TODOS os dados que você pediu, exportada para o seu computador
nome_arquivo = 'Tabela_Bruta_TB_HIV_TeleSAP.csv'
df_pacientes[['Record ID', 'Grupo', 'Status_HIV']].to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
print(f"Sucesso! A tabela completa (linha a linha por paciente) foi salva como '{nome_arquivo}'.\n")

# ==============================================================================
# 4. AGREGAÇÃO MATEMÁTICA PARA O GRÁFICO
# ==============================================================================
# Calcula a porcentagem de HIV dentro de cada grupo de Tuberculose
df_agregado = df_pacientes.groupby('Grupo')['Tem_HIV'].mean() * 100
df_agregado = df_agregado.reset_index()
df_agregado.rename(columns={'Tem_HIV': 'Prevalência_HIV'}, inplace=True)

# Garante a ordem correta para bater com a sua imagem
ordem = ['Sem Tuberculose', 'Com Tuberculose']
df_agregado['Grupo'] = pd.Categorical(df_agregado['Grupo'], categories=ordem, ordered=True)
df_agregado = df_agregado.sort_values('Grupo')

# ==============================================================================
# 5. RECRIAÇÃO DO GRÁFICO (MATPLOTLIB + SEABORN)
# ==============================================================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

cores = ['#4a8ec2', '#d05c53']

ax = sns.barplot(
    data=df_agregado, 
    x='Grupo', 
    y='Prevalência_HIV', 
    palette=cores
)

plt.title('Prevalência Co-infectiva: HIV/AIDS em Pacientes com Tuberculose\n(Dados Extraídos da Base TeleSAP)', 
          fontsize=15, pad=20)
plt.xlabel('Grupo', fontsize=12)
plt.ylabel('Prevalência de HIV (%)', fontsize=12)

# Ajusta o eixo Y (o máximo será um pouco maior que a barra mais alta para caber o texto)
plt.ylim(0, max(df_agregado['Prevalência_HIV']) * 1.3)

for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', 
                xytext=(0, 5), textcoords='offset points', 
                fontweight='bold', fontsize=12, color='#2c3e50')

sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

# ==============================================================================
# 6. RELATÓRIO DO CONSOLE (N ABSOLUTO)
# ==============================================================================
print(f"{'='*80}")
print("SÍNTESE DOS DADOS REAIS (TELESAP)")
print(f"{'='*80}")
total_sem_tb = len(df_pacientes[df_pacientes['Tem_TB'] == 0])
hiv_sem_tb = df_pacientes[df_pacientes['Tem_TB'] == 0]['Tem_HIV'].sum()

total_com_tb = len(df_pacientes[df_pacientes['Tem_TB'] == 1])
hiv_com_tb = df_pacientes[df_pacientes['Tem_TB'] == 1]['Tem_HIV'].sum()

print(f" -> PACIENTES SEM TUBERCULOSE: {total_sem_tb} (Destes, {hiv_sem_tb} têm HIV)")
print(f" -> PACIENTES COM TUBERCULOSE: {total_com_tb} (Destes, {hiv_com_tb} têm HIV)")
print(f"{'='*80}\n")

In [ ]:
# ==============================================================================
# VISUALIZAÇÃO AVANÇADA: FLUXO EPIDEMIOLÓGICO POR PACIENTE ÚNICO (N=238) E TABELA
# ==============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO E MAPEAMENTO DE COLUNAS
# ------------------------------------------------------------------------------
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Localizando as colunas do REDCap dinamicamente
col_tb = [c for c in df.columns if 'choice=Tuberculose' in c and 'infecciosas' in c.lower()][0]
col_hiv = [c for c in df.columns if 'choice=HIV/AIDS' in c and 'infecciosas' in c.lower()][0]
cols_espec = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]

# ------------------------------------------------------------------------------
# 2. IDENTIFICAÇÃO DE PACIENTES ÚNICOS (TRAVA 1 PESSOA = 1 LINHA)
# ------------------------------------------------------------------------------
# Pegamos uma lista apenas dos IDs (Record IDs) que marcaram HIV alguma vez
ids_hiv_positivos = df[df[col_hiv] == 'Checked']['Record ID'].unique()

registros_para_grafico = []

# Varremos o histórico completo de cada um desses pacientes
for paciente_id in ids_hiv_positivos:
    # Isolamos todas as linhas (cadastro e consultas) apenas deste paciente
    hist_paciente = df[df['Record ID'] == paciente_id]

    # 1. Verifica se em alguma linha ele teve Tuberculose marcada
    teve_tb = 'TB Positivo' if (hist_paciente[col_tb] == 'Checked').any() else 'TB Negativo'
    coinfeccao = 1 if teve_tb == 'TB Positivo' else 0

    # 2. Verifica TODAS as especialidades para as quais ele foi encaminhado na vida
    especs_do_paciente = set()
    for col_esp in cols_espec:
        if (hist_paciente[col_esp] == 'Checked').any():
            nome_limpo = col_esp.split('choice=')[1].replace(')', '').strip()
            especs_do_paciente.add(nome_limpo)

    # 3. Define o destino final do paciente para o gráfico
    if len(especs_do_paciente) == 0:
        destino_final = 'Sem Encaminhamento (Resolvido na APS)'
    elif len(especs_do_paciente) == 1:
        destino_final = list(especs_do_paciente)[0]
    else:
        destino_final = 'Múltiplos Encaminhamentos (Histórico)'

    # Adiciona o PACIENTE (e não a consulta) ao gráfico
    registros_para_grafico.append({
        'Record ID': paciente_id,
        'Status_HIV': 'HIV Positivo',
        'Status_TB': teve_tb,
        'Risco_Cor': coinfeccao,
        'Especialidade': destino_final
    })

df_plot = pd.DataFrame(registros_para_grafico)

# ------------------------------------------------------------------------------
# 3. OTIMIZAÇÃO E GERAÇÃO DO DIAGRAMA PLOTLY
# ------------------------------------------------------------------------------
if not df_plot.empty:
    # Agrupamos especialidades menos frequentes em "Outras"
    top_especialidades = df_plot['Especialidade'].value_counts().nlargest(10).index
    df_plot['Especialidade_Plot'] = np.where(df_plot['Especialidade'].isin(top_especialidades),
                                             df_plot['Especialidade'],
                                             'Outras Especialidades')

    # Criação do Gráfico
    fig = px.parallel_categories(
        df_plot,
        dimensions=['Status_HIV', 'Status_TB', 'Especialidade_Plot'],
        color="Risco_Cor",
        color_continuous_scale=[(0.0, '#3498db'), (1.0, '#e74c3c')], # Azul (Apenas HIV) | Vermelho (HIV + TB)
        labels={
            'Status_HIV': 'Coorte Base (Pacientes HIV+)',
            'Status_TB': 'Diagnóstico de TB',
            'Especialidade_Plot': 'Destino Clínico do Paciente'
        },
        title=f"Jornada Clínica do Paciente HIV+ (Total: {len(df_plot)} Pacientes Únicos)"
    )

    # Melhorando o design
    fig.update_layout(
        font=dict(size=13, family="Arial"),
        coloraxis_showscale=False,
        margin=dict(l=50, r=50, t=80, b=50)
    )

    # Exibe o gráfico interativo
    fig.show()

    # ------------------------------------------------------------------------------
    # 4. GERAÇÃO E EXPORTAÇÃO DA TABELA EXATA DO GRÁFICO (PACIENTES ÚNICOS)
    # ------------------------------------------------------------------------------
    # Agrupa as 3 dimensões do gráfico para contar o número de pacientes em cada "linha" do diagrama
    df_tabela = df_plot.groupby(
        ['Status_HIV', 'Status_TB', 'Especialidade_Plot']
    ).size().reset_index(name='Quantidade_Pacientes')

    # Ordena a tabela para facilitar a leitura (Coinfectados primeiro, depois por volume)
    df_tabela = df_tabela.sort_values(
        by=['Status_TB', 'Quantidade_Pacientes'], 
        ascending=[False, False]
    ).reset_index(drop=True)

    # Imprime no terminal de forma elegante
    print("\n" + "="*85)
    print(f"TABELA DE FLUXO: JORNADA CLÍNICA DO PACIENTE HIV+ (N={len(df_plot)} Pacientes Únicos)")
    print("="*85)
    print(df_tabela.to_string(index=False))
    print("="*85)

    # Exporta para um arquivo CSV no seu computador
    nome_arquivo = 'Tabela_Fluxo_Jornada_HIV_Pacientes_Unicos.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
    print(f"\n[SUCESSO] A tabela completa do gráfico foi salva como '{nome_arquivo}' na sua pasta.")

else:
    print("Nenhum paciente HIV Positivo encontrado para gerar o gráfico.")

In [ ]:
# ==============================================================================
# VISUALIZAÇÃO ESTATÍSTICA: EVOLUÇÃO ETÁRIA POR CIAP (POPULAÇÃO HIV+)
# ==============================================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Localizando as colunas
col_idade = 'Idade'
col_hiv = [c for c in df.columns if 'choice=HIV/AIDS' in c and 'infecciosas' in c.lower()][0]

cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
if not cols_ciap:
    raise ValueError("Nenhuma coluna de CIAP encontrada.")
col_ciap = cols_ciap[0]

# ==============================================================================
# 2. PREPARAÇÃO DOS DADOS (Focando na População HIV+)
# ==============================================================================
df['Idade_Num'] = pd.to_numeric(df[col_idade], errors='coerce')

# Propagamos a Idade e o status de HIV do cadastro para todas as consultas do paciente
df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
df[['Idade_Num', col_hiv]] = df.groupby('Record ID')[['Idade_Num', col_hiv]].ffill()

# Filtramos APENAS as consultas realizadas
df_ev = df[df['Repeat Instrument'].notna()].copy()

# Filtramos APENAS os pacientes HIV Positivos
df_hiv = df_ev[df_ev[col_hiv] == 'Checked'].copy()

# Limpeza: removemos consultas sem Idade ou sem CIAP preenchido
df_hiv = df_hiv.dropna(subset=['Idade_Num', col_ciap])
df_hiv = df_hiv[df_hiv[col_ciap].astype(str).str.strip() != '']
df_hiv = df_hiv[df_hiv[col_ciap].astype(str).str.lower() != 'nan']

if df_hiv.empty:
    print("Nenhuma consulta válida encontrada para pacientes com HIV (ou falta preenchimento de CIAP/Idade).")
else:
    # Limpamos o texto do CIAP para não quebrar a tela
    # Usamos o apply com strip para garantir que não haja espaços vazios atrapalhando
    df_hiv['CIAP_Clean'] = df_hiv[col_ciap].astype(str).str.strip().apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

    # ==============================================================================
    # 3. RELATÓRIO EXECUTIVO: QUAIS CIAPS MAIS APARECEM?
    # ==============================================================================
    print("\n" + "="*80)
    print(f"🦠 PERFIL DIAGNÓSTICO (POPULAÇÃO HIV POSITIVA: {len(df_hiv)} consultas válidas)")
    print("="*80)

    ranking_ciap = df_hiv['CIAP_Clean'].value_counts()
    for ciap, qtd in ranking_ciap.head(10).items():
        pct = (qtd / len(df_hiv)) * 100
        print(f" - {str(ciap)[:50]:<50}: {qtd} ocorrências ({pct:.1f}%)")
    print("="*80)

    # ==============================================================================
    # 4. GERAÇÃO DO RIDGELINE PLOT (IDADE vs CIAP)
    # ==============================================================================
    # Pegamos os Top 8 CIAPs para fazer as montanhas
    top_8_ciaps = ranking_ciap.nlargest(8).index
    df_plot = df_hiv[df_hiv['CIAP_Clean'].isin(top_8_ciaps)].copy()

    fig = go.Figure()

    # Paleta de cores em tons de vermelho/quentes para destacar a população alvo
    colors = px.colors.sequential.Reds[::-1] # Invertido para os mais escuros ficarem no topo

    # Lemos de trás para frente para a montanha #1 ficar no fundo (topo visual)
    for i, ciap in enumerate(reversed(top_8_ciaps)):
        idade_array = df_plot[df_plot['CIAP_Clean'] == ciap]['Idade_Num']
        cor_atual = colors[(i + 2) % len(colors)] # +2 para pular as cores muito claras da paleta

        fig.add_trace(go.Violin(
            x=idade_array,
            name=ciap,
            side='positive',      # Apenas metade superior
            width=1.5,            # Largura controlada para não sobrepor demais
            line_color='black',   # Contorno
            line_width=1.2,
            fillcolor=cor_atual,
            opacity=0.85,
            points=False,
            meanline_visible=True,# Linha da média ativada
            spanmode='hard'       # Corta a curva nos limites reais de idade do grupo
        ))

    # ==============================================================================
    # 5. LAYOUT FINAL DO GRÁFICO
    # ==============================================================================
    fig.update_layout(
        title=dict(
            text=f"Evolução Etária dos Principais CIAPs (População HIV+, N={len(df_hiv)} consultas)",
            font=dict(size=18, family="Arial")
        ),
        xaxis=dict(
            title="Idade do Paciente (Anos)",
            showgrid=True,
            gridcolor='#e0e0e0',
            zeroline=False,
            title_font=dict(size=14)
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False
        ),
        height=700,
        margin=dict(l=300, r=50, t=80, b=80), # Margem esquerda grande para caber os nomes dos CIAPs
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()

    # ==============================================================================
    # 6. GERAÇÃO E EXPORTAÇÃO DA TABELA ESTATÍSTICA (AS BASES DAS MONTANHAS)
    # ==============================================================================
    df_tabela = df_plot.groupby('CIAP_Clean')['Idade_Num'].agg(
        Quantidade_Consultas='count',
        Idade_Minima='min',
        Idade_Media='mean',
        Idade_Mediana='median',
        Idade_Maxima='max'
    ).reset_index()

    # Arredondamos as idades médias para 1 casa decimal
    df_tabela['Idade_Media'] = df_tabela['Idade_Media'].round(1)

    # Ordenamos pelo volume de consultas (do maior para o menor)
    df_tabela = df_tabela.sort_values(by='Quantidade_Consultas', ascending=False).reset_index(drop=True)

    print("\n" + "="*95)
    print("TABELA ESTATÍSTICA: DISTRIBUIÇÃO DE IDADE POR CIAP (Top 8 - População HIV+)")
    print("="*95)
    print(df_tabela.to_string(index=False))
    print("="*95)

    nome_arquivo = 'Tabela_Ridgeline_Idades_CIAP_HIV.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
    print(f"\n[SUCESSO] A tabela estatística do gráfico foi salva como '{nome_arquivo}'.")